In [ ]:
# Install blosc2 and caterva2 in Pyodide environments (automatically added)
import sys

if sys.platform == "emscripten":
    import micropip
    import requests

    # Install latest blosc2
    blosc_latest_url = "https://blosc.github.io/python-blosc2/wheels/latest.txt"
    blosc_wheel_name = requests.get(blosc_latest_url).text.strip()
    blosc_wheel_url = f"https://blosc.github.io/python-blosc2/wheels/{blosc_wheel_name}"
    await micropip.install(blosc_wheel_url)
    print(f"Installed {blosc_wheel_name} successfully!")

    # Install latest caterva2
    caterva_latest_url = "https://ironarray.github.io/Caterva2/wheels/latest.txt"
    caterva_wheel_name = requests.get(caterva_latest_url).text.strip()
    caterva_wheel_url = f"https://ironarray.github.io/Caterva2/wheels/{caterva_wheel_name}"
    await micropip.install(caterva_wheel_url)
    print(f"Installed {caterva_wheel_name} successfully!")

In [ ]:
import json

import caterva2 as cat2

# In JupyterLite/Pyodide, Caterva2 can use the same origin as the notebook.
client = cat2.Client(None)

session = None
session_id = None


def new_agent_session(name="prova3", notebook_path="prova3.ipynb"):
    global session, session_id
    session = client.create_llm_session(name=name, notebook_path=notebook_path)
    session_id = session["session_id"]
    print(f"LLM session ready: {session_id}")
    return session


def ask(message, *, show_trace=False):
    if not session_id:
        raise RuntimeError("No LLM session yet. Run new_agent_session() first.")

    response = client.chat_llm(session_id, message)
    print(response["assistant"]["text"])

    if show_trace:
        print("\nTRACE:")
        print(json.dumps(response["trace"], indent=2))

    return response


def reset_agent_session():
    if not session_id:
        raise RuntimeError("No LLM session yet. Run new_agent_session() first.")
    result = client.reset_llm_session(session_id)
    print(f"LLM session reset: {result['session_id']}")
    return result


def delete_agent_session():
    global session, session_id
    if not session_id:
        raise RuntimeError("No LLM session yet. Run new_agent_session() first.")
    result = client.delete_llm_session(session_id)
    print(f"LLM session deleted: {result['session_id']}")
    session = None
    session_id = None
    return result


new_agent_session()

In [ ]:
# Example prompts
# ask("List the available roots")
# ask("List datasets under @public/dir1")
# ask("Show metadata for @public/ds-1d.b2nd")
# ask("Show stats for @public/ds-1d.b2nd", show_trace=True)
